In [1]:
# 11 GLM v3 + claim history, GBM + history, blend: two-fold validation
# Purpose: add the claim-history features (src/history.py) and measure the gain on the same folds.
# Rules fixed before results:
#  (a) history is adopted in the GLM if average validation D2 rises by >= 0.0005 and the A/E of rows
#      with a past claim moves toward 1;
#  (b) GBM or blend is adopted only if >= 0.003 above GLM v3 + history, with lowest/highest decile A/E
#      no more than 0.1 further from 1 than the GLM's.
import sys, time
sys.path.append("..")
import numpy as np
import pandas as pd
import lightgbm as lgb
from src.target import LINES, load_policies, load_claims, build_model_target
from src.features import build_features
from src.history import claim_history
from src.glm import BandedTweedieGLM
from src.evaluate import evaluate
from src.diagnostics import FOLDS

NON_FEATURES = ("row_id", "year", "exposure_clean", "target", "weight")
DROP = {"motor": ["previous_insurer", "customer_tenure"],
        "buildings": ["previous_insurer", "occupation", "region", "customer_tenure"],
        "contents": ["previous_insurer", "occupation", "customer_tenure"]}
HIST = ["hist_1y", "hist_2y"]
GLM_CONFIG = {"motor": dict(alpha=0.5, n_bands=10, drop=None, balance=True),
              "buildings": dict(alpha=0.5, n_bands=20, drop=None, balance=True),
              "contents": dict(alpha=0.3, n_bands=20, drop=None, balance=True)}
PARAMS2 = dict(objective="tweedie", tweedie_variance_power=1.5, metric="tweedie", learning_rate=0.03,
               num_leaves=31, min_data_in_leaf=500, feature_fraction=0.8, bagging_fraction=0.8,
               bagging_freq=1, lambda_l2=1.0, min_data_per_group=1000, cat_smooth=50, cat_l2=10,
               max_cat_threshold=8, verbose=-1, seed=42)


def hist_levels(n):
    """Counts -> '0', '1', '2+' ; unknown history stays missing (its own level in the GLM)."""
    return pd.Series(np.select([n == 0, n == 1, n >= 2], ["0", "1", "2+"], default=None), index=n.index,
                     dtype="object").where(n.notna())


def add_history(F, p_rows, p_all_same_line, claims):
    h = claim_history(p_rows, p_all_same_line, claims)
    h["hist_1y"] = hist_levels(h.pol_claims_1y)
    h["hist_2y"] = hist_levels(h.pol_claims_2y)
    out = F.merge(h[["row_id"] + HIST], on="row_id", how="left", validate="one_to_one")
    assert len(out) == len(F)
    return out


def prep2(X, cats=None):
    X = X.copy()
    if cats is None:
        cats = {c: sorted(X[c].dropna().astype(str).unique()) for c in X.columns
                if not pd.api.types.is_numeric_dtype(X[c]) and not pd.api.types.is_bool_dtype(X[c])}
    for c, levels in cats.items():
        s = X[c].astype("string")
        X[c] = pd.Categorical(s.where(s.isin(levels)), categories=levels)
    for c in X.columns:
        if pd.api.types.is_bool_dtype(X[c]):
            X[c] = X[c].astype(int)
    return X, cats


def fit_gbm2(tr, feats, params=PARAMS2, patience=300):
    last = tr.year.max()
    inner, hold = tr[tr.year < last], tr[tr.year == last]
    Xi, cats = prep2(inner[feats]); Xh, _ = prep2(hold[feats], cats)
    di = lgb.Dataset(Xi, inner.target / inner.weight, weight=inner.weight)
    dh = lgb.Dataset(Xh, hold.target / hold.weight, weight=hold.weight, reference=di)
    b = lgb.train(params, di, num_boost_round=5000, valid_sets=[dh],
                  callbacks=[lgb.early_stopping(patience, verbose=False)])
    n_trees = max(int(b.best_iteration * 1.1), 1)
    Xt, cats = prep2(tr[feats])
    model = lgb.train(params, lgb.Dataset(Xt, tr.target / tr.weight, weight=tr.weight), num_boost_round=n_trees)
    k = tr.target.sum() / (model.predict(Xt) * tr.weight).sum()
    return model, cats, k, n_trees


# 1. Modelling tables with history (train); test history uses train + test periods of the same line
claims = load_claims()
data = {}
for line in LINES:
    p, pt = load_policies(line, "train"), load_policies(line, "test")
    mt, comp, loading = build_model_target(claims, p, line)
    F = add_history(build_features(line, "train", policies=p), p, [p, pt], claims)
    df = mt.merge(F, on="row_id", how="left", validate="one_to_one")
    data[line] = {"df": df, "loading": loading}
    print(f"{line}: hist_1y shares by year\n",
          df.groupby("year").hist_1y.value_counts(normalize=True, dropna=False).unstack().round(3).to_string())

# 2. Two-fold validation
rows, hist_ae = [], []
for line in LINES:
    df = data[line]["df"]
    base = [c for c in df.columns if c not in NON_FEATURES and c not in DROP[line] and c not in HIST]
    withh = base + HIST
    for vy, tys in FOLDS:
        t0 = time.time()
        tr, va = df[df.year.isin(tys)], df[df.year == vy]
        out = {}
        for name, feats in [("GLM v3", base), ("GLM v3 + history", withh)]:
            glm = BandedTweedieGLM(**GLM_CONFIG[line]).fit(tr[feats], tr.target / tr.weight, tr.weight)
            out[name] = glm.predict_rate(va[feats]) * va.weight.to_numpy()
        model, cats, k, n_trees = fit_gbm2(tr, withh)
        Xv, _ = prep2(va[withh], cats)
        out["GBM + history"] = k * model.predict(Xv) * va.weight.to_numpy()
        out["blend + history"] = 0.5 * out["GLM v3 + history"] + 0.5 * out["GBM + history"]
        for name, pred in out.items():
            r = evaluate(va.target, pred, va.weight, show=False)
            lf = r["lift"]["A/E"]
            rows.append({"line": line, "year": vy, "model": name, "D2": r["D2"], "A/E": r["A/E"],
                         "low": lf.iloc[0], "top": lf.iloc[-1]})
            g = pd.DataFrame({"h": va.hist_1y.fillna("unknown").values, "a": va.target.values, "p": pred})
            s = g.groupby("h")[["a", "p"]].sum()
            for lvl, ae in (s.a / s.p).items():
                hist_ae.append({"line": line, "year": vy, "model": name, "hist_1y": lvl, "A/E": ae})
        print(f"{line} {vy}: GBM {n_trees} trees, k = {k:.3f}, {time.time() - t0:.0f}s")

res = pd.DataFrame(rows)
res.to_csv("../results/history_validation.csv", index=False)
for line in LINES:
    t = res[res.line == line].pivot(index="model", columns="year", values=["D2", "A/E", "low", "top"])
    t.columns = [f"{m}_{y}" for m, y in t.columns]
    t["D2_avg"] = t[["D2_2023", "D2_2024"]].mean(axis=1)
    print(f"\n===== {line} =====")
    print(t.sort_values("D2_avg", ascending=False).round(4).to_string())

ha = pd.DataFrame(hist_ae).pivot_table(index=["line", "model"], columns=["hist_1y", "year"], values="A/E")
print("\nA/E by previous-year claims on the same policy (should move toward 1 with history):")
print(ha.round(3).to_string())

avg = res.groupby(["line", "model"]).D2.mean().unstack()
print("\nrule (a): history in the GLM (gain vs GLM v3; adopt if >= 0.0005):")
for line in LINES:
    d = avg.loc[line, "GLM v3 + history"] - avg.loc[line, "GLM v3"]
    print(f"  {line:9s} {d:+.4f}  {'adopt' if d >= 0.0005 else 'do not adopt'}")
print("\nrule (b): GBM / blend vs GLM v3 + history (D2 threshold 0.003; check tails in the tables):")
for line in LINES:
    for m in ["GBM + history", "blend + history"]:
        d = avg.loc[line, m] - avg.loc[line, "GLM v3 + history"]
        print(f"  {line:9s} {m:16s} {d:+.4f}  {'meets D2 threshold' if d >= 0.003 else 'below'}")

motor: hist_1y shares by year
 hist_1y      0      1     2+    NaN
year                               
2021       NaN    NaN    NaN  1.000
2022     0.754  0.084  0.011  0.152
2023     0.755  0.082  0.010  0.153
2024     0.757  0.080  0.009  0.154
buildings: hist_1y shares by year
 hist_1y      0      1     2+    NaN
year                               
2021       NaN    NaN    NaN  1.000
2022     0.815  0.031  0.001  0.153
2023     0.816  0.030  0.001  0.153
2024     0.816  0.031  0.001  0.152
contents: hist_1y shares by year
 hist_1y      0      1     2+    NaN
year                               
2021       NaN    NaN    NaN  1.000
2022     0.817  0.028  0.001  0.154
2023     0.818  0.028  0.001  0.153
2024     0.818  0.029  0.001  0.152
motor 2023: GBM 157 trees, k = 1.089, 46s
motor 2024: GBM 207 trees, k = 1.081, 70s
buildings 2023: GBM 78 trees, k = 1.170, 26s
buildings 2024: GBM 126 trees, k = 1.149, 38s
contents 2023: GBM 82 trees, k = 1.167, 27s
contents 2024: GBM 126 trees, k =

In [2]:
# 11b Final fit on 2021-2024: GLM v4 (= v3 + claim history) and an experimental blend
from pathlib import Path
from src.target import DATA_DIR

subs_glm, subs_blend, checks = [], [], []
for line in LINES:
    df, loading = data[line]["df"], data[line]["loading"]
    feats = [c for c in df.columns if c not in NON_FEATURES and c not in DROP[line]]   # includes HIST
    glm = BandedTweedieGLM(**GLM_CONFIG[line]).fit(df[feats], df.target / df.weight, df.weight)

    p, pt = load_policies(line, "train"), load_policies(line, "test")
    Ft = add_history(build_features(line, "test", policies=pt), pt, [p, pt], claims)
    assert list(Ft.row_id) == list(pt.row_id)
    expo = pt.exposure_clean.to_numpy()
    rate_glm = glm.predict_rate(Ft[feats])
    pred_glm = np.maximum((rate_glm + loading) * expo, 0)
    subs_glm.append(pd.DataFrame({"row_id": Ft.row_id, "prediction": pred_glm}))

    if line == "motor":
        pred_blend = pred_glm
    else:
        model, cats, k, n_trees = fit_gbm2(df, feats)
        Xt, _ = prep2(Ft[feats], cats)
        pred_blend = np.maximum((0.5 * rate_glm + 0.5 * k * model.predict(Xt) + loading) * expo, 0)
    subs_blend.append(pd.DataFrame({"row_id": Ft.row_id, "prediction": pred_blend}))

    d24 = df[df.year == 2024]
    lvl24 = np.average(glm.predict_rate(d24[feats]), weights=d24.exposure_clean)
    checks.append({"line": line, "glm_k": glm.k_,
                   "hist_1y test (0/1/2+/unknown)": Ft.hist_1y.fillna("unk").value_counts(normalize=True)
                       .reindex(["0", "1", "2+", "unk"]).round(3).tolist(),
                   "hist_1y train 2024": d24.hist_1y.fillna("unk").value_counts(normalize=True)
                       .reindex(["0", "1", "2+", "unk"]).round(3).tolist(),
                   "glm test/2024": np.average(rate_glm, weights=expo) / lvl24,
                   "blend/glm total": pred_blend.sum() / pred_glm.sum()})
    print(f"{line}: done")

pd.set_option("display.max_colwidth", 60)
print(pd.DataFrame(checks).set_index("line").to_string())

sample = pd.read_parquet(DATA_DIR / "sample_submission.parquet")
out = Path("../submissions"); out.mkdir(exist_ok=True)
for name, parts in [("glm_v4.csv", subs_glm), ("blend_v2_experiment.csv", subs_blend)]:
    sub = pd.concat(parts, ignore_index=True)
    assert len(sub) == len(sample) and set(sub.row_id) == set(sample.row_id) and sub.row_id.is_unique
    assert sub.prediction.notna().all() and (sub.prediction >= 0).all()
    sub = sample[["row_id"]].merge(sub, on="row_id", how="left")
    sub.to_csv(out / name, index=False)
    print(f"wrote {name}: total {sub.prediction.sum():,.0f}")

motor: done
buildings: done
contents: done
              glm_k hist_1y test (0/1/2+/unknown)            hist_1y train 2024  glm test/2024  blend/glm total
line                                                                                                           
motor      1.019120   [0.759, 0.08, 0.009, 0.152]   [0.757, 0.08, 0.009, 0.154]       0.953806         1.000000
buildings  1.040111  [0.816, 0.031, 0.001, 0.151]  [0.816, 0.031, 0.001, 0.152]       0.985234         0.994027
contents   1.051660  [0.819, 0.029, 0.001, 0.152]  [0.818, 0.029, 0.001, 0.152]       1.012552         0.993051
wrote glm_v4.csv: total 190,249,631
wrote blend_v2_experiment.csv: total 189,705,569


In [3]:
# 11c Unattended experiments. Run AFTER the 11 cell (uses data, add_history, prep2, PARAMS2, GLM_CONFIG, DROP ...).
# Every result is appended to ../results/overnight/ as soon as it exists, so a crash keeps what was done.
#  1. NCB / bonus-hunger check (model-free): do frequency and severity move in opposite directions with NCB?
#  2. Method B: GBM fitted on top of GLM v4 (GLM log-prediction as init_score), default GBM settings.
#  3. GBM settings grid (plain GBM and method B). The setting is chosen on the held-out LAST TRAINING YEAR
#     only; the validation year is used once, for the chosen setting.
# Rules fixed before results (same as before): adopt GBM / method B only if average validation D2 is >= 0.003
# above GLM v4 and lowest/highest decile A/E are no more than 0.1 further from 1 than GLM v4's.
import time, traceback
from pathlib import Path
from itertools import product

OUT = Path("../results/overnight"); OUT.mkdir(parents=True, exist_ok=True)


def log(msg):
    line_ = f"{time.strftime('%H:%M:%S')} {msg}"
    print(line_, flush=True)
    with open(OUT / "log.txt", "a") as f:
        f.write(line_ + "\n")


_ROWS = {}


def append_csv(row, name):
    """Keep all rows in memory and rewrite the whole file, so rows with different columns stay aligned."""
    _ROWS.setdefault(name, []).append(row)
    pd.DataFrame(_ROWS[name]).to_csv(OUT / name, index=False)


def metrics(va, pred):
    r = evaluate(va.target, pred, va.weight, show=False)
    lf = r["lift"]["A/E"]
    return {"D2": r["D2"], "A/E": r["A/E"], "low": lf.iloc[0], "top": lf.iloc[-1]}


def feats_of(df, line):
    return [c for c in df.columns if c not in NON_FEATURES and c not in DROP[line]]


# ---------------------------------------------------------------- 1. NCB check
def ncb_check():
    p = load_policies("motor", "train")
    m = match_claims(claims, p, "motor", cap=None)
    m = m[m.loss_year <= 2023]                                   # mature years only
    df = data["motor"]["df"]
    df = df[df.year <= 2023]
    band = lambda s: pd.cut(s, [-0.5, 0.5, 2.5, 4.5, 8.5, 100], labels=["0", "1-2", "3-4", "5-8", "9+"])
    expo = df.assign(b=band(df.ncb_years)).groupby("b", observed=True).exposure_clean.sum()
    cm = m.merge(df[["row_id", "ncb_years"]], on="row_id", how="inner")
    cm["b"] = band(cm.ncb_years)
    rows = []
    for peril in ["own_damage", "all perils"]:
        s = cm if peril == "all perils" else cm[cm.peril == peril]
        g = s.groupby("b", observed=True)
        t = pd.DataFrame({"freq_per_1000": g.size() / expo * 1000,
                          "mean_gross": g.incurred_to_date.apply(lambda x: np.minimum(x, 500_000).mean()),
                          "small_claim_share": g.apply(lambda x: (x.incurred_to_date <= x.excess + 500).mean()),
                          "exposure_share": expo / expo.sum()})
        t["peril"] = peril
        rows.append(t.reset_index())
    res = pd.concat(rows)
    res.to_csv(OUT / "1_ncb_check.csv", index=False)
    log("1. NCB check (2021-23):\n" + res.round(3).to_string(index=False))


# ------------------------------------------------- shared GLM fits per line and fold
def glm_parts(df, line, vy, tys, feats):
    tr, va = df[df.year.isin(tys)], df[df.year == vy]
    last = tr.year.max()
    inner, hold = tr[tr.year < last], tr[tr.year == last]
    g_in = BandedTweedieGLM(**GLM_CONFIG[line]).fit(inner[feats], inner.target / inner.weight, inner.weight)
    g_tr = BandedTweedieGLM(**GLM_CONFIG[line]).fit(tr[feats], tr.target / tr.weight, tr.weight)
    return dict(tr=tr, va=va, inner=inner, hold=hold,
                li=np.log(g_in.predict_rate(inner[feats])), lh=np.log(g_in.predict_rate(hold[feats])),
                lt=np.log(g_tr.predict_rate(tr[feats])), lv=np.log(g_tr.predict_rate(va[feats])))


def fit_gbm(P, feats, params, offset, patience=300):
    """Plain GBM (offset=False) or method B (offset=True: start from the GLM log-prediction).
    Trees chosen on the held-out last training year; refit on all training years; balance factor k."""
    inner, hold, tr, va = P["inner"], P["hold"], P["tr"], P["va"]
    Xi, cats = prep2(inner[feats]); Xh, _ = prep2(hold[feats], cats)
    di = lgb.Dataset(Xi, inner.target / inner.weight, weight=inner.weight, init_score=P["li"] if offset else None)
    dh = lgb.Dataset(Xh, hold.target / hold.weight, weight=hold.weight, init_score=P["lh"] if offset else None,
                     reference=di)
    b = lgb.train(params, di, num_boost_round=5000, valid_sets=[dh],
                  callbacks=[lgb.early_stopping(patience, verbose=False)])
    inner_score = b.best_score["valid_0"]["tweedie"]
    n = max(int(b.best_iteration * 1.1), 1)
    Xt, cats = prep2(tr[feats]); Xv, _ = prep2(va[feats], cats)
    model = lgb.train(params, lgb.Dataset(Xt, tr.target / tr.weight, weight=tr.weight,
                                          init_score=P["lt"] if offset else None), num_boost_round=n)
    if offset:
        rate_tr = np.exp(P["lt"] + model.predict(Xt, raw_score=True))
        rate_va = np.exp(P["lv"] + model.predict(Xv, raw_score=True))
    else:
        rate_tr, rate_va = model.predict(Xt), model.predict(Xv)
    k = tr.target.sum() / (rate_tr * tr.weight).sum()
    return k * rate_va * va.weight.to_numpy(), dict(trees=n, inner_best=b.best_iteration,
                                                    inner_score=inner_score, k=k)


# ----------------------------------------------------- 2 and 3. method B and the grid
GRID = [dict(num_leaves=nl, min_data_in_leaf=md) for nl, md in product([15, 31, 63], [200, 1000, 3000])]
DEFAULT = dict(num_leaves=31, min_data_in_leaf=500)


def run_models(only_default):
    configs = [DEFAULT] if only_default else GRID
    fname = "2_method_b_default.csv" if only_default else "3_grid.csv"
    for line in LINES:
        df = data[line]["df"]
        feats = feats_of(df, line)
        for vy, tys in FOLDS:
            t0 = time.time()
            P = glm_parts(df, line, vy, tys, feats)
            va = P["va"]
            glm_pred = np.exp(P["lv"]) * va.weight.to_numpy()
            append_csv({"line": line, "year": vy, "model": "GLM v4", "config": "", **metrics(va, glm_pred)}, fname)
            for cfg, offset in product(configs, [False, True]):
                name = "method B" if offset else "GBM"
                try:
                    pred, info = fit_gbm(P, feats, dict(PARAMS2, **cfg), offset)
                    row = {"line": line, "year": vy, "model": name, "config": str(cfg), **info, **metrics(va, pred)}
                    append_csv(row, fname)
                    if not offset:
                        append_csv({"line": line, "year": vy, "model": "blend 50/50", "config": str(cfg),
                                    **metrics(va, 0.5 * glm_pred + 0.5 * pred)}, fname)
                    log(f"{fname} {line} {vy} {name} {cfg}: trees {info['trees']}, k {info['k']:.3f}, "
                        f"D2 {row['D2']:.4f}")
                except Exception:
                    log(f"FAILED {line} {vy} {name} {cfg}\n{traceback.format_exc()}")
            log(f"{fname} {line} {vy} done in {(time.time() - t0) / 60:.1f} min")


def summarise_grid():
    g = pd.read_csv(OUT / "3_grid.csv")
    rows = []
    for line in LINES:
        glm = g[(g.line == line) & (g.model == "GLM v4")].groupby("year").D2.mean()
        for name in ["GBM", "method B"]:
            s = g[(g.line == line) & (g.model == name)]
            pick = s.groupby("config").inner_score.mean().idxmin()           # chosen on training years only
            c = s[s.config == pick].set_index("year")
            rows.append({"line": line, "model": name, "chosen": pick, "trees": c.trees.round().tolist(),
                         "k": c.k.round(3).tolist(), "D2_avg": c.D2.mean(), "gain_vs_GLM": c.D2.mean() - glm.mean(),
                         "low": c.low.round(3).tolist(), "top": c.top.round(3).tolist()})
            if name == "GBM":
                b = g[(g.line == line) & (g.model == "blend 50/50") & (g.config == pick)].set_index("year")
                rows.append({"line": line, "model": "blend 50/50", "chosen": pick, "D2_avg": b.D2.mean(),
                             "gain_vs_GLM": b.D2.mean() - glm.mean(), "low": b.low.round(3).tolist(),
                             "top": b.top.round(3).tolist()})
        gl = g[(g.line == line) & (g.model == "GLM v4")].set_index("year")
        rows.append({"line": line, "model": "GLM v4", "D2_avg": gl.D2.mean(), "gain_vs_GLM": 0.0,
                     "low": gl.low.round(3).tolist(), "top": gl.top.round(3).tolist()})
    s = pd.DataFrame(rows)
    s.to_csv(OUT / "3_grid_summary.csv", index=False)
    log("3. grid summary (setting chosen on the held-out last training year):\n" + s.round(4).to_string(index=False))


from src.target import match_claims
log("=== start ===")
for step, fn in [("1 NCB check", ncb_check), ("2 method B (default settings)", lambda: run_models(True)),
                 ("3 settings grid", lambda: run_models(False)), ("3 summary", summarise_grid)]:
    try:
        log(f"--- {step} ---")
        fn()
    except Exception:
        log(f"STEP FAILED: {step}\n{traceback.format_exc()}")
log("=== all done ===")

12:55:31 === start ===
12:55:31 --- 1 NCB check ---
12:55:31 1. NCB check (2021-23):
  b  freq_per_1000  mean_gross  small_claim_share  exposure_share      peril
  0        140.630    2173.353              0.371           0.065 own_damage
1-2         92.219    2189.115              0.377           0.179 own_damage
3-4         67.116    2364.442              0.352           0.136 own_damage
5-8         50.514    2451.700              0.337           0.209 own_damage
 9+         41.684    2523.606              0.327           0.410 own_damage
  0        299.113    4268.036              0.310           0.065 all perils
1-2        207.186    4285.796              0.312           0.179 all perils
3-4        152.486    4305.879              0.318           0.136 all perils
5-8        118.675    3838.451              0.324           0.209 all perils
 9+        101.965    3924.684              0.326           0.410 all perils
12:55:31 --- 2 method B (default settings) ---
12:56:17 2_method_b_d

In [4]:
# 11d Monotone calibration of GBM / blend (home lines; motor for reference). Run in the 11 kernel after 11c.
# Problem: GBM and the blend over-price the lowest decile (A/E 0.56-0.74), which fails our tail rule.
# Fix tested: a monotone curve "predicted rate -> actual rate", fitted on the HELD-OUT LAST TRAINING YEAR with
# models trained on the earlier training years, then applied to the final model's predictions and rebalanced
# on the training years. The validation year is not used to build anything.
# Rule fixed before results: adopt the calibrated GBM or blend for a line if average validation D2 >= GLM v4
# + 0.003 and, in each year, lowest and highest decile A/E are no more than 0.1 further from 1 than GLM v4's.
from sklearn.isotonic import IsotonicRegression

CHOSEN = {"motor": dict(num_leaves=15, min_data_in_leaf=3000),       # chosen in 11c on the last training year
          "buildings": dict(num_leaves=15, min_data_in_leaf=1000),
          "contents": dict(num_leaves=15, min_data_in_leaf=3000)}


def gbm_hold_and_final(P, feats, params, patience=300):
    """GBM on inner years (trees by early stopping on the hold year) -> hold-year rates for calibration;
    then the final GBM on all training years -> training and validation rates (balanced)."""
    inner, hold, tr, va = P["inner"], P["hold"], P["tr"], P["va"]
    Xi, cats = prep2(inner[feats]); Xh, _ = prep2(hold[feats], cats)
    di = lgb.Dataset(Xi, inner.target / inner.weight, weight=inner.weight)
    dh = lgb.Dataset(Xh, hold.target / hold.weight, weight=hold.weight, reference=di)
    b = lgb.train(params, di, num_boost_round=5000, valid_sets=[dh],
                  callbacks=[lgb.early_stopping(patience, verbose=False)])
    k_in = inner.target.sum() / (b.predict(Xi, num_iteration=b.best_iteration) * inner.weight).sum()
    hold_rate = k_in * b.predict(Xh, num_iteration=b.best_iteration)
    n = max(int(b.best_iteration * 1.1), 1)
    Xt, cats = prep2(tr[feats]); Xv, _ = prep2(va[feats], cats)
    m = lgb.train(params, lgb.Dataset(Xt, tr.target / tr.weight, weight=tr.weight), num_boost_round=n)
    rt, rv = m.predict(Xt), m.predict(Xv)
    k = tr.target.sum() / (rt * tr.weight).sum()
    return hold_rate, k * rt, k * rv, n


def fit_calibration(pred_rate, actual_cost, weight, n_bins=50):
    """Monotone map of predicted rate -> actual rate, on weighted quantile bins (in log scale)."""
    lx = np.log(np.maximum(pred_rate, 1e-9))
    order = np.argsort(lx)
    cw = np.cumsum(weight[order]) / weight.sum()
    b = np.minimum((cw * n_bins).astype(int), n_bins - 1)
    df_ = pd.DataFrame({"b": b, "lx": lx[order], "w": weight[order], "a": actual_cost[order],
                        "p": pred_rate[order] * weight[order]})
    g = df_.groupby("b").apply(lambda s: pd.Series({"lx": np.average(s.lx, weights=s.w), "w": s.w.sum(),
                                                    "a_rate": s.a.sum() / s.w.sum()}))
    iso = IsotonicRegression(increasing=True, out_of_bounds="clip", y_min=1e-9)
    fitted = iso.fit_transform(g.lx, g.a_rate, sample_weight=g.w)
    xs, ys = g.lx.to_numpy(), np.log(np.maximum(fitted, 1e-9))
    return lambda rate: np.exp(np.interp(np.log(np.maximum(rate, 1e-9)), xs, ys))


def tails_ok(m, ref):
    return all(abs(m[c] - 1) <= abs(ref[c] - 1) + 0.1 for c in ["low", "top"])


rows = []
for line in LINES:
    df = data[line]["df"]
    feats = feats_of(df, line)
    params = dict(PARAMS2, **CHOSEN[line])
    for vy, tys in FOLDS:
        t0 = time.time()
        P = glm_parts(df, line, vy, tys, feats)
        tr, va, hold = P["tr"], P["va"], P["hold"]
        g_hold, g_tr, g_va = np.exp(P["lh"]), np.exp(P["lt"]), np.exp(P["lv"])
        b_hold, b_tr, b_va, n = gbm_hold_and_final(P, feats, params)
        cands = {"GLM v4": (None, g_tr, g_va),
                 "GBM": (b_hold, b_tr, b_va),
                 "blend": (0.5 * g_hold + 0.5 * b_hold, 0.5 * g_tr + 0.5 * b_tr, 0.5 * g_va + 0.5 * b_va)}
        out = {}
        for name, (h, rtr, rva) in cands.items():
            out[name] = rva * va.weight.to_numpy()
            if h is not None:
                f = fit_calibration(h, hold.target.to_numpy(), hold.weight.to_numpy())
                ctr, cva = f(rtr), f(rva)
                k2 = tr.target.sum() / (ctr * tr.weight).sum()                 # rebalance on training years
                out[name + " calibrated"] = k2 * cva * va.weight.to_numpy()
        for name, pred in out.items():
            rows.append({"line": line, "year": vy, "model": name, **metrics(va, pred)})
        log(f"11d {line} {vy}: GBM {n} trees, {time.time() - t0:.0f}s")

cal = pd.DataFrame(rows)
cal.to_csv(OUT / "4_calibration.csv", index=False)
for line in LINES:
    t = cal[cal.line == line].pivot(index="model", columns="year", values=["D2", "A/E", "low", "top"])
    t.columns = [f"{m}_{y}" for m, y in t.columns]
    t["D2_avg"] = t[["D2_2023", "D2_2024"]].mean(axis=1)
    print(f"\n===== {line} =====")
    print(t.sort_values("D2_avg", ascending=False).round(4).to_string())

print("\nrule: D2 >= GLM v4 + 0.003 and tails no more than 0.1 further from 1 than GLM v4 (each year)")
for line in LINES:
    c = cal[cal.line == line].set_index(["model", "year"])
    base = c.loc["GLM v4"]
    for m in ["GBM", "GBM calibrated", "blend", "blend calibrated"]:
        gain = c.loc[m].D2.mean() - base.D2.mean()
        tails = all(tails_ok(c.loc[(m, y)], base.loc[y]) for y in [2023, 2024])
        verdict = "ADOPT" if gain >= 0.003 and tails else "no"
        print(f"  {line:9s} {m:17s} gain {gain:+.4f} | tails {'ok' if tails else 'fail'} -> {verdict}")

15:25:23 11d motor 2023: GBM 259 trees, 93s
15:27:54 11d motor 2024: GBM 283 trees, 151s
15:28:41 11d buildings 2023: GBM 101 trees, 47s
15:30:01 11d buildings 2024: GBM 174 trees, 80s
15:30:49 11d contents 2023: GBM 108 trees, 49s
15:32:14 11d contents 2024: GBM 193 trees, 84s

===== motor =====
                  D2_2023  D2_2024  A/E_2023  A/E_2024  low_2023  low_2024  top_2023  top_2024  D2_avg
model                                                                                                 
blend              0.0543   0.0558    1.0565    1.0224    0.8857    0.9894    1.1139    1.0943  0.0551
blend calibrated   0.0529   0.0550    1.0628    1.0257    0.9236    1.1555    0.9868    1.0732  0.0540
GBM                0.0529   0.0544    1.0858    1.0434    0.9785    1.0473    1.1362    1.0858  0.0537
GBM calibrated     0.0519   0.0543    1.0900    1.0471    0.9780    1.2634    1.0496    1.0626  0.0531
GLM v4             0.0518   0.0543    1.0288    1.0022    0.9121    0.9795    1.0312

In [5]:
# 11e v5 = v4 with contents replaced by the GLM v4 + GBM 50/50 blend (passed the pre-set rule in 11d)
line = "contents"
df, loading = data[line]["df"], data[line]["loading"]
feats = feats_of(df, line)
glm = BandedTweedieGLM(**GLM_CONFIG[line]).fit(df[feats], df.target / df.weight, df.weight)
model, cats, k, n_trees = fit_gbm2(df, feats, params=dict(PARAMS2, **CHOSEN[line]))

p, pt = load_policies(line, "train"), load_policies(line, "test")
Ft = add_history(build_features(line, "test", policies=pt), pt, [p, pt], claims)
assert list(Ft.row_id) == list(pt.row_id)
Xt, _ = prep2(Ft[feats], cats)
rate = 0.5 * glm.predict_rate(Ft[feats]) + 0.5 * k * model.predict(Xt)
pred = np.maximum((rate + loading) * pt.exposure_clean.to_numpy(), 0)

v4 = pd.read_csv("../submissions/glm_v4.csv")
sub = v4.set_index("row_id")
old = sub.loc[Ft.row_id, "prediction"].to_numpy()
sub.loc[Ft.row_id, "prediction"] = pred
sub = sub.reset_index()
assert len(sub) == len(v4) and (sub.row_id == v4.row_id).all()
assert sub.prediction.notna().all() and (sub.prediction >= 0).all()
print(f"contents: {n_trees} trees, k {k:.3f}; contents total v5 / v4 = {pred.sum() / old.sum():.3f}; "
      f"rows changed by > 2x: {((pred / old > 2) | (pred / old < 0.5)).mean():.2%}")
sub.to_csv("../submissions/v5.csv", index=False)
print("wrote submissions/v5.csv")

contents: 269 trees, k 1.087; contents total v5 / v4 = 0.998; rows changed by > 2x: 0.02%
wrote submissions/v5.csv


In [2]:
# 11 setup: imports, functions and modelling tables with claim history (no experiments).
# Run this after a kernel restart, then 11f. Takes 2-3 minutes.
import sys, time
sys.path.append("..")
from pathlib import Path
import numpy as np
import pandas as pd
import lightgbm as lgb
from src.target import LINES, load_policies, load_claims, build_model_target
from src.features import build_features
from src.history import claim_history
from src.glm import BandedTweedieGLM
from src.evaluate import evaluate
from src.diagnostics import FOLDS

NON_FEATURES = ("row_id", "year", "exposure_clean", "target", "weight")
DROP = {"motor": ["previous_insurer", "customer_tenure"],
        "buildings": ["previous_insurer", "occupation", "region", "customer_tenure"],
        "contents": ["previous_insurer", "occupation", "customer_tenure"]}
HIST = ["hist_1y", "hist_2y"]
GLM_CONFIG = {"motor": dict(alpha=0.5, n_bands=10, drop=None, balance=True),
              "buildings": dict(alpha=0.5, n_bands=20, drop=None, balance=True),
              "contents": dict(alpha=0.3, n_bands=20, drop=None, balance=True)}
PARAMS2 = dict(objective="tweedie", tweedie_variance_power=1.5, metric="tweedie", learning_rate=0.03,
               num_leaves=31, min_data_in_leaf=500, feature_fraction=0.8, bagging_fraction=0.8,
               bagging_freq=1, lambda_l2=1.0, min_data_per_group=1000, cat_smooth=50, cat_l2=10,
               max_cat_threshold=8, verbose=-1, seed=42)
CHOSEN = {"motor": dict(num_leaves=15, min_data_in_leaf=3000),
          "buildings": dict(num_leaves=15, min_data_in_leaf=1000),
          "contents": dict(num_leaves=15, min_data_in_leaf=3000)}
OUT = Path("../results/overnight"); OUT.mkdir(parents=True, exist_ok=True)


def hist_levels(n):
    """Counts -> '0', '1', '2+' ; unknown history stays missing (its own level in the GLM)."""
    return pd.Series(np.select([n == 0, n == 1, n >= 2], ["0", "1", "2+"], default=None), index=n.index,
                     dtype="object").where(n.notna())


def add_history(F, p_rows, p_all_same_line, claims):
    h = claim_history(p_rows, p_all_same_line, claims)
    h["hist_1y"] = hist_levels(h.pol_claims_1y)
    h["hist_2y"] = hist_levels(h.pol_claims_2y)
    out = F.merge(h[["row_id"] + HIST], on="row_id", how="left", validate="one_to_one")
    assert len(out) == len(F)
    return out


def prep2(X, cats=None):
    X = X.copy()
    if cats is None:
        cats = {c: sorted(X[c].dropna().astype(str).unique()) for c in X.columns
                if not pd.api.types.is_numeric_dtype(X[c]) and not pd.api.types.is_bool_dtype(X[c])}
    for c, levels in cats.items():
        s = X[c].astype("string")
        X[c] = pd.Categorical(s.where(s.isin(levels)), categories=levels)
    for c in X.columns:
        if pd.api.types.is_bool_dtype(X[c]):
            X[c] = X[c].astype(int)
    return X, cats


def fit_gbm2(tr, feats, params=PARAMS2, patience=300):
    last = tr.year.max()
    inner, hold = tr[tr.year < last], tr[tr.year == last]
    Xi, cats = prep2(inner[feats]); Xh, _ = prep2(hold[feats], cats)
    di = lgb.Dataset(Xi, inner.target / inner.weight, weight=inner.weight)
    dh = lgb.Dataset(Xh, hold.target / hold.weight, weight=hold.weight, reference=di)
    b = lgb.train(params, di, num_boost_round=5000, valid_sets=[dh],
                  callbacks=[lgb.early_stopping(patience, verbose=False)])
    n_trees = max(int(b.best_iteration * 1.1), 1)
    Xt, cats = prep2(tr[feats])
    model = lgb.train(params, lgb.Dataset(Xt, tr.target / tr.weight, weight=tr.weight), num_boost_round=n_trees)
    k = tr.target.sum() / (model.predict(Xt) * tr.weight).sum()
    return model, cats, k, n_trees


def log(msg):
    line_ = f"{time.strftime('%H:%M:%S')} {msg}"
    print(line_, flush=True)
    with open(OUT / "log.txt", "a") as f:
        f.write(line_ + "\n")


def metrics(va, pred):
    r = evaluate(va.target, pred, va.weight, show=False)
    lf = r["lift"]["A/E"]
    return {"D2": r["D2"], "A/E": r["A/E"], "low": lf.iloc[0], "top": lf.iloc[-1]}


def feats_of(df, line):
    return [c for c in df.columns if c not in NON_FEATURES and c not in DROP[line]]


claims = load_claims()
data = {}
for line in LINES:
    p, pt = load_policies(line, "train"), load_policies(line, "test")
    mt, comp, loading = build_model_target(claims, p, line)
    F = add_history(build_features(line, "train", policies=p), p, [p, pt], claims)
    df = mt.merge(F, on="row_id", how="left", validate="one_to_one")
    data[line] = {"df": df, "loading": loading}
    print(f"{line}: table ready")
print("setup done")

motor: table ready
buildings: table ready
contents: table ready
setup done


In [3]:
# 11f Claim history by peril group (counts only; previous year).
# Rule fixed before results: adopt in the GLM for a line if average validation D2 rises by >= 0.0005 over
# GLM v4 and the A/E of rows with a past claim in the main group moves toward 1.
GROUPS = {"motor": {"bi": ["bodily_injury"], "od": ["own_damage"], "tpp": ["third_party_prop"],
                    "oth": ["glass_hail", "theft"]},
          "buildings": {"wthr": ["flood", "storm", "subsidence"], "eow": ["escape_of_water"],
                        "oth": ["fire", "accidental_damage"]},
          "contents": {"theft": ["theft"], "eow": ["escape_of_water"], "oth": ["flood", "accidental_damage"]}}
MAIN = {"motor": "hist_bi_1y", "buildings": "hist_wthr_1y", "contents": "hist_theft_1y"}


def add_peril_history(F, p_rows, p_all, claims, line):
    out = F.copy()
    for g, perils in GROUPS[line].items():
        h = claim_history(p_rows, p_all, claims, years=(1,), perils=perils, prefix=f"hist_{g}")
        col = f"hist_{g}_1y"
        lvl = pd.Series(np.where(h[col] >= 1, "1+", "0"), index=h.index, dtype="object").where(h[col].notna())
        out = out.merge(pd.DataFrame({"row_id": h.row_id, col: lvl}), on="row_id", how="left", validate="one_to_one")
    assert len(out) == len(F)
    return out


PH = {}
for line in LINES:
    p, pt = load_policies(line, "train"), load_policies(line, "test")
    df2 = add_peril_history(data[line]["df"], p, [p, pt], claims, line)
    PH[line] = [f"hist_{g}_1y" for g in GROUPS[line]]
    data[line]["df_ph"] = df2
    print(f"{line}: share with >= 1 claim last year by group (2024): " +
          ", ".join(f"{c} {(df2[df2.year == 2024][c] == '1+').mean():.1%}" for c in PH[line]))

rows, ae = [], []
for line in LINES:
    df2 = data[line]["df_ph"]
    base = [c for c in df2.columns if c not in NON_FEATURES and c not in DROP[line] and c not in PH[line]]
    for vy, tys in FOLDS:
        t0 = time.time()
        tr, va = df2[df2.year.isin(tys)], df2[df2.year == vy]
        for name, feats in [("GLM v4", base), ("GLM v4 + peril history", base + PH[line])]:
            glm = BandedTweedieGLM(**GLM_CONFIG[line]).fit(tr[feats], tr.target / tr.weight, tr.weight)
            pred = glm.predict_rate(va[feats]) * va.weight.to_numpy()
            rows.append({"line": line, "year": vy, "model": name, **metrics(va, pred)})
            g = pd.DataFrame({"h": va[MAIN[line]].fillna("unknown").values, "a": va.target.values, "p": pred})
            s = g.groupby("h")[["a", "p"]].sum()
            for lvl, v in (s.a / s.p).items():
                ae.append({"line": line, "year": vy, "model": name, "main_group": lvl, "A/E": v})
        log(f"11f {line} {vy}: {time.time() - t0:.0f}s")

ph = pd.DataFrame(rows)
ph.to_csv(OUT / "5_peril_history.csv", index=False)
for line in LINES:
    t = ph[ph.line == line].pivot(index="model", columns="year", values=["D2", "A/E", "low", "top"])
    t.columns = [f"{m}_{y}" for m, y in t.columns]
    t["D2_avg"] = t[["D2_2023", "D2_2024"]].mean(axis=1)
    print(f"\n===== {line} =====")
    print(t.round(4).to_string())
print("\nA/E by previous-year claims in the main peril group (motor: BI; buildings: weather; contents: theft):")
print(pd.DataFrame(ae).pivot_table(index=["line", "model"], columns=["main_group", "year"], values="A/E")
      .round(3).to_string())
avg = ph.groupby(["line", "model"]).D2.mean().unstack()
print("\nrule: adopt if GLM D2 gain >= 0.0005 (and the main-group A/E moves toward 1):")
for line in LINES:
    d = avg.loc[line, "GLM v4 + peril history"] - avg.loc[line, "GLM v4"]
    print(f"  {line:9s} {d:+.4f}  {'adopt (check A/E table)' if d >= 0.0005 else 'do not adopt'}")

motor: share with >= 1 claim last year by group (2024): hist_bi_1y 0.5%, hist_od_1y 4.1%, hist_tpp_1y 2.9%, hist_oth_1y 2.1%
buildings: share with >= 1 claim last year by group (2024): hist_wthr_1y 1.4%, hist_eow_1y 1.2%, hist_oth_1y 0.6%
contents: share with >= 1 claim last year by group (2024): hist_theft_1y 1.0%, hist_eow_1y 0.7%, hist_oth_1y 1.4%
20:10:11 11f motor 2023: 40s
20:11:12 11f motor 2024: 61s
20:11:34 11f buildings 2023: 21s
20:12:05 11f buildings 2024: 31s
20:12:28 11f contents 2023: 23s
20:13:02 11f contents 2024: 34s

===== motor =====
                        D2_2023  D2_2024  A/E_2023  A/E_2024  low_2023  low_2024  top_2023  top_2024  D2_avg
model                                                                                                       
GLM v4                   0.0518   0.0543    1.0288    1.0022    0.9121    0.9795    1.0312    1.0473   0.053
GLM v4 + peril history   0.0519   0.0542    1.0254    1.0015    0.9120    0.9879    1.0188    1.0396   0.053

===

In [4]:
# 11g (1) feature clean-up and (2) motor large-loss loading allocation. Run in the 11 kernel after 11_setup.
# Rules fixed before results:
#  (1) drop a feature set if average validation D2 falls by no more than 0.0005 (simpler model preferred);
#      add age_at_licence (replacing years_licensed) only if D2 rises by >= 0.0005.
#  (2) loading proportional to predicted risk is adopted only if D2 on UNCAPPED actual cost is higher than
#      the flat loading in BOTH validation years.
from src.target import ADJ, adjust_claims, match_claims

HOMEISH = ["soil_type", "roof_type", "energy_rating", "building_age", "property_type", "construction_type",
           "bedrooms", "flood_zone", "elevation_m", "distance_to_river_km", "year_built", "occupancy"]

rows, load_rows = [], []
for line in LINES:
    df = data[line]["df"].copy()
    base = feats_of(df, line)
    variants = {"GLM v4": base}
    if "discount_applied" in base:
        variants["drop discount_applied"] = [c for c in base if c != "discount_applied"]
    if line == "motor":
        home = [c for c in HOMEISH if c in base]
        print("home-type features found in motor:", home)
        variants["drop home-type features"] = [c for c in base if c not in home]
        df["age_at_licence"] = df["driver_age"] - df["years_licensed"]
        variants["age_at_licence for years_licensed"] = [c for c in base if c != "years_licensed"] + ["age_at_licence"]
        p = load_policies("motor", "train")
        mc = adjust_claims(match_claims(claims, p, "motor", cap=None), ADJ["motor"])
        df["uncapped"] = df.row_id.map(mc.groupby("row_id").net_adj_uncapped.sum()).fillna(0.0)
    for vy, tys in FOLDS:
        t0 = time.time()
        tr, va = df[df.year.isin(tys)], df[df.year == vy]
        for name, feats in variants.items():
            glm = BandedTweedieGLM(**GLM_CONFIG[line]).fit(tr[feats], tr.target / tr.weight, tr.weight)
            rate_va = glm.predict_rate(va[feats])
            rows.append({"line": line, "year": vy, "model": name, **metrics(va, rate_va * va.weight.to_numpy())})
            if line == "motor" and name == "GLM v4":
                rate_tr = glm.predict_rate(tr[feats])
                L = (tr.uncapped - tr.target).sum() / tr.weight.sum()           # loading from training years
                mean_rate = np.average(rate_tr, weights=tr.weight)
                w = va.weight.to_numpy()
                va_u = va.assign(target=va.uncapped)                            # evaluate on uncapped cost
                for lname, pred in [("flat loading (current)", (rate_va + L) * w),
                                    ("loading proportional to predicted rate", rate_va * (1 + L / mean_rate) * w)]:
                    load_rows.append({"year": vy, "method": lname, "loading_per_weight": L, **metrics(va_u, pred)})
        log(f"11g {line} {vy}: {time.time() - t0:.0f}s")

fx = pd.DataFrame(rows)
fx.to_csv(OUT / "6_features.csv", index=False)
for line in LINES:
    t = fx[fx.line == line].pivot(index="model", columns="year", values=["D2", "A/E", "low", "top"])
    t.columns = [f"{m}_{y}" for m, y in t.columns]
    t["D2_avg"] = t[["D2_2023", "D2_2024"]].mean(axis=1)
    print(f"\n===== {line} =====")
    print(t.sort_values("D2_avg", ascending=False).round(4).to_string())
avg = fx.groupby(["line", "model"]).D2.mean().unstack()
print("\nrule (1): drop if D2 falls by <= 0.0005; add age_at_licence if D2 rises by >= 0.0005")
for line in LINES:
    for m in [c for c in avg.columns if c != "GLM v4" and pd.notna(avg.loc[line, c])]:
        d = avg.loc[line, m] - avg.loc[line, "GLM v4"]
        ok = d >= 0.0005 if m.startswith("age_at") else d >= -0.0005
        print(f"  {line:9s} {m:36s} {d:+.4f}  {'adopt' if ok else 'keep as is'}")

ld = pd.DataFrame(load_rows)
ld.to_csv(OUT / "7_loading.csv", index=False)
print("\n(2) motor loading on UNCAPPED actual cost:")
print(ld.pivot(index="method", columns="year", values=["D2", "A/E", "low", "top"]).round(4).to_string())
d = ld.pivot(index="method", columns="year", values="D2")
better = (d.loc["loading proportional to predicted rate"] > d.loc["flat loading (current)"]).all()
print(f"rule (2): proportional loading better in both years -> {'adopt' if better else 'keep flat loading'}")

home-type features found in motor: ['soil_type', 'roof_type', 'energy_rating', 'building_age', 'property_type', 'bedrooms', 'flood_zone', 'elevation_m', 'distance_to_river_km']
20:20:43 11g motor 2023: 77s
20:22:38 11g motor 2024: 115s
20:22:58 11g buildings 2023: 21s
20:23:30 11g buildings 2024: 32s
20:23:54 11g contents 2023: 23s
20:24:27 11g contents 2024: 34s

===== motor =====
                                   D2_2023  D2_2024  A/E_2023  A/E_2024  low_2023  low_2024  top_2023  top_2024  D2_avg
model                                                                                                                  
drop home-type features             0.0522   0.0546    1.0295    1.0026    0.9100    0.9839    1.0394    1.0505  0.0534
drop discount_applied               0.0518   0.0543    1.0288    1.0022    0.9139    0.9786    1.0294    1.0475  0.0530
GLM v4                              0.0518   0.0543    1.0288    1.0022    0.9121    0.9795    1.0312    1.0473  0.0530
age_at_licence 

In [5]:
# 11h Motor large-loss loading from a fitted tail (Pareto) instead of the observed excess alone.
# Why: the current loading (about 16 per weight) comes almost entirely from one open 2023 bodily injury claim
# (incurred 11.9m), so it depends on whether that one event recurs. A tail fitted to all large claims gives
# an expected cost above the 500k cap that one claim cannot dominate.
# Pareto above a threshold u: P(X > x | X > u) = (u / x)^alpha; alpha by maximum likelihood = n / sum(log(X / u)).
# Expected cost above the cap c per claim over u: E[(X - c)+ | X > u] = u^alpha * c^(1 - alpha) / (alpha - 1).
# Loading per weight = (claims over u per weight) * that expected cost.
# Rule fixed before results: use the Pareto loading (u = 100k) if the estimates for u = 100k, 150k, 200k are
# within +-30% of each other and alpha > 1.2 at all three; otherwise keep the observed loading and record the
# uncertainty.
from src.target import ADJ, adjust_claims, match_claims

CAP = ADJ["motor"]["cap"]
p = load_policies("motor", "train")
mc = adjust_claims(match_claims(claims, p, "motor", cap=None), ADJ["motor"])
x_all = mc.net_adj_uncapped.to_numpy()                       # 2025 level, net of excess, uncapped
W = data["motor"]["df"].weight.sum()                         # exposure x completeness, 2021-24

obs = (mc.net_adj_uncapped - mc.net_adj).sum() / W
top = mc.net_adj_uncapped.max()
obs_wo_top = (mc.net_adj_uncapped - mc.net_adj).sum() - (top - min(top, CAP))
print(f"observed loading: {obs:.2f} per weight; without the single largest claim ({top:,.0f}): "
      f"{obs_wo_top / W:.2f}")
print(f"claims above the cap: {(x_all > CAP).sum()}; largest five: {np.sort(x_all)[-5:].round(0).tolist()}\n")

rows = []
for u in [50_000, 100_000, 150_000, 200_000]:
    x = x_all[x_all > u]
    n = len(x)
    alpha = n / np.log(x / u).sum()
    se = alpha / np.sqrt(n)
    excess = u ** alpha * CAP ** (1 - alpha) / (alpha - 1) if alpha > 1 else np.inf
    rows.append({"threshold": u, "claims_over_u": n, "alpha": alpha, "alpha_se": se,
                 "freq_over_u_per_weight": n / W, "E[(X-cap)+ | X>u]": excess,
                 "loading_per_weight": n / W * excess})
    rows[-1]["share_over_cap_obs"] = (x > CAP).mean()
    rows[-1]["share_over_cap_pareto"] = (u / CAP) ** alpha
tl = pd.DataFrame(rows)
tl.to_csv(OUT / "8_tail_loading.csv", index=False)
pd.set_option("display.width", 200)
print(tl.round(4).to_string(index=False))

ref = tl.set_index("threshold").loc[[100_000, 150_000, 200_000]]
spread = ref.loading_per_weight.max() / ref.loading_per_weight.min() - 1
stable = (ref.loading_per_weight / ref.loading_per_weight.loc[100_000]).between(0.7, 1.3).all() and (ref.alpha > 1.2).all()
print(f"\nrule: estimates within +-30% of the u=100k value and alpha > 1.2 -> "
      f"{'use Pareto loading ' + format(ref.loading_per_weight.loc[100_000], '.2f') if stable else 'keep observed loading'} "
      f"(spread across thresholds {spread:.0%})")

observed loading: 16.00 per weight; without the single largest claim (11,894,027): 0.71
claims above the cap: 3; largest five: [434718.0, 475577.0, 601550.0, 928410.0, 11894027.0]

 threshold  claims_over_u  alpha  alpha_se  freq_over_u_per_weight  E[(X-cap)+ | X>u]  loading_per_weight  share_over_cap_obs  share_over_cap_pareto
     50000            461 1.9415    0.0904                  0.0006          6076.8310              3.7595              0.0065                 0.0114
    100000            119 2.0468    0.1876                  0.0002         17720.5451              2.8299              0.0252                 0.0371
    150000             53 1.9633    0.2697                  0.0001         48823.3512              3.4726              0.0566                 0.0941
    200000             31 1.9783    0.3553                  0.0000         83420.3118              3.4705              0.0968                 0.1632

rule: estimates within +-30% of the u=100k value and alpha > 1.2 -> use P

In [6]:
# 11i v6: motor without home-type features and discount_applied, large-loss loading 2.83 (Pareto, u = 100k)
# allocated in proportion to predicted rate; buildings and contents without discount_applied (contents keeps
# the GLM + LightGBM 50/50 blend). All changes passed rules fixed before their results (11g, 11h).
from pathlib import Path
from src.target import DATA_DIR

HOME_IN_MOTOR = ["soil_type", "roof_type", "energy_rating", "building_age", "property_type", "bedrooms",
                 "flood_zone", "elevation_m", "distance_to_river_km"]
EXTRA_DROP = {"motor": HOME_IN_MOTOR + ["discount_applied"], "buildings": ["discount_applied"],
              "contents": ["discount_applied"]}
MOTOR_LOADING = 2.83                                       # 11h, Pareto tail above 100k

v5 = pd.read_csv("../submissions/v5.csv").set_index("row_id")
parts, checks = [], []
for line in LINES:
    df, loading = data[line]["df"], data[line]["loading"]
    feats = [c for c in feats_of(df, line) if c not in EXTRA_DROP[line]]
    glm = BandedTweedieGLM(**GLM_CONFIG[line]).fit(df[feats], df.target / df.weight, df.weight)
    p, pt = load_policies(line, "train"), load_policies(line, "test")
    Ft = add_history(build_features(line, "test", policies=pt), pt, [p, pt], claims)
    assert list(Ft.row_id) == list(pt.row_id)
    expo = pt.exposure_clean.to_numpy()
    rate = glm.predict_rate(Ft[feats])
    if line == "contents":
        model, cats, k, n_trees = fit_gbm2(df, feats, params=dict(PARAMS2, **CHOSEN[line]))
        Xt, _ = prep2(Ft[feats], cats)
        rate = 0.5 * rate + 0.5 * k * model.predict(Xt)
    if line == "motor":
        mean_rate = np.average(glm.predict_rate(df[feats]), weights=df.weight)
        pred = np.maximum(rate * (1 + MOTOR_LOADING / mean_rate) * expo, 0)
        used = f"{MOTOR_LOADING} proportional (x{1 + MOTOR_LOADING / mean_rate:.4f})"
    else:
        pred = np.maximum((rate + loading) * expo, 0)
        used = f"{loading:.2f} flat"
    parts.append(pd.DataFrame({"row_id": Ft.row_id, "prediction": pred}))
    old = v5.loc[Ft.row_id, "prediction"].to_numpy()
    checks.append({"line": line, "features": len(feats), "loading": used,
                   "total v6/v5": pred.sum() / old.sum(),
                   "rows changed > 2x": ((pred / old > 2) | (pred / old < 0.5)).mean()})
    print(f"{line}: done")

print(pd.DataFrame(checks).set_index("line").round(4).to_string())
sample = pd.read_parquet(DATA_DIR / "sample_submission.parquet")
sub = pd.concat(parts, ignore_index=True)
assert len(sub) == len(sample) and set(sub.row_id) == set(sample.row_id) and sub.row_id.is_unique
assert sub.prediction.notna().all() and (sub.prediction >= 0).all()
sub = sample[["row_id"]].merge(sub, on="row_id", how="left")
sub.to_csv("../submissions/v6.csv", index=False)
print(f"wrote submissions/v6.csv: total {sub.prediction.sum():,.0f}")

motor: done
buildings: done
contents: done
           features                      loading  total v6/v5  rows changed > 2x
line                                                                            
motor            48  2.83 proportional (x1.0054)       0.9730                0.0
buildings        45                    2.33 flat       1.0003                0.0
contents         47                    0.00 flat       0.9987                0.0
wrote submissions/v6.csv: total 187,463,333


In [7]:
# 11j Sensitivity: Pareto tail without the single largest claim
x_wo = np.sort(x_all)[:-1]
for u in [100_000, 150_000, 200_000]:
    x = x_wo[x_wo > u]; n = len(x)
    a = n / np.log(x / u).sum()
    load = n / W * u ** a * CAP ** (1 - a) / (a - 1)
    print(f"u={u:>7,}: n={n:3d}, alpha {a:.3f} (with the claim: {tl.set_index('threshold').alpha[u]:.3f}), "
          f"loading {load:.2f} (with: {tl.set_index('threshold').loading_per_weight[u]:.2f})")

u=100,000: n=118, alpha 2.211 (with the claim: 2.047), loading 1.86 (with: 2.83)
u=150,000: n= 52, alpha 2.299 (with the claim: 1.963), loading 1.69 (with: 3.47)
u=200,000: n= 30, alpha 2.590 (with the claim: 1.978), loading 1.18 (with: 3.47)


In [8]:
# 11k Overnight: more regularised LightGBM, blend weight and seed bagging, all chosen on the held-out last
# training year; validation used once per line. Run in the 11 kernel after 11_setup (and 11i for EXTRA_DROP).
# Rule fixed before results: adopt for a line if average validation D2 >= GLM (v6 features) + 0.003 and, in each
# year, lowest and highest decile A/E are no more than 0.1 further from 1 than the GLM's.
# Results are written to ../results/overnight/9_*.csv as they come; log in log.txt.
import traceback
from itertools import product
from sklearn.metrics import mean_tweedie_deviance

EXTRA_DROP = {"motor": ["soil_type", "roof_type", "energy_rating", "building_age", "property_type", "bedrooms",
                        "flood_zone", "elevation_m", "distance_to_river_km", "discount_applied"],
              "buildings": ["discount_applied"], "contents": ["discount_applied"]}
GRID = [dict(num_leaves=nl, min_data_in_leaf=md, lambda_l2=l2)
        for nl, md, l2 in product([7, 15], [1000, 5000, 15000], [1.0, 10.0, 50.0])]
WEIGHTS = [0.75, 0.5, 0.25]                                   # weight on the GLM in the blend
SEEDS = [42, 7, 2026]
_R = {}


def save(row, name):
    _R.setdefault(name, []).append(row)
    pd.DataFrame(_R[name]).to_csv(OUT / name, index=False)


def hold_dev(actual, rate, w):
    return mean_tweedie_deviance(actual / w, np.maximum(rate, 1e-9), sample_weight=w, power=1.5)


def glm_fits(df, line, vy, tys, feats):
    tr, va = df[df.year.isin(tys)], df[df.year == vy]
    last = tr.year.max()
    inner, hold = tr[tr.year < last], tr[tr.year == last]
    g_in = BandedTweedieGLM(**GLM_CONFIG[line]).fit(inner[feats], inner.target / inner.weight, inner.weight)
    g_tr = BandedTweedieGLM(**GLM_CONFIG[line]).fit(tr[feats], tr.target / tr.weight, tr.weight)
    return dict(tr=tr, va=va, inner=inner, hold=hold, g_hold=g_in.predict_rate(hold[feats]),
                g_va=g_tr.predict_rate(va[feats]))


def gbm(P, feats, params, seed):
    """Trees by early stopping on the hold year; returns hold-year rate (inner model), the hold deviance,
    and the validation rate of the final model on all training years (balanced)."""
    prm = dict(params, seed=seed)
    inner, hold, tr, va = P["inner"], P["hold"], P["tr"], P["va"]
    Xi, cats = prep2(inner[feats]); Xh, _ = prep2(hold[feats], cats)
    di = lgb.Dataset(Xi, inner.target / inner.weight, weight=inner.weight)
    dh = lgb.Dataset(Xh, hold.target / hold.weight, weight=hold.weight, reference=di)
    b = lgb.train(prm, di, num_boost_round=5000, valid_sets=[dh], callbacks=[lgb.early_stopping(300, verbose=False)])
    k_in = inner.target.sum() / (b.predict(Xi, num_iteration=b.best_iteration) * inner.weight).sum()
    h_rate = k_in * b.predict(Xh, num_iteration=b.best_iteration)
    n = max(int(b.best_iteration * 1.1), 1)
    Xt, cats = prep2(tr[feats]); Xv, _ = prep2(va[feats], cats)
    m = lgb.train(prm, lgb.Dataset(Xt, tr.target / tr.weight, weight=tr.weight), num_boost_round=n)
    k = tr.target.sum() / (m.predict(Xt) * tr.weight).sum()
    return h_rate, k * m.predict(Xv), n, k


def tails_ok(m, ref):
    return all(abs(m[c] - 1) <= abs(ref[c] - 1) + 0.1 for c in ["low", "top"])


log("=== 11k start ===")
summary = []
for line in LINES:
    try:
        df = data[line]["df"]
        feats = [c for c in feats_of(df, line) if c not in EXTRA_DROP[line]]
        P = {vy: glm_fits(df, line, vy, tys, feats) for vy, tys in FOLDS}
        glm_m = {vy: metrics(P[vy]["va"], P[vy]["g_va"] * P[vy]["va"].weight.to_numpy()) for vy in P}
        # 1. settings grid (seed 42): hold deviance of the GBM and of each blend weight, plus validation metrics
        res = {}
        for cfg in GRID:
            params = dict(PARAMS2, **cfg)
            for vy in P:
                t0 = time.time()
                h, v, n, k = gbm(P[vy], feats, params, SEEDS[0])
                hold = P[vy]["hold"]
                for wgt in [0.0] + WEIGHTS:
                    hr = wgt * P[vy]["g_hold"] + (1 - wgt) * h
                    vr = wgt * P[vy]["g_va"] + (1 - wgt) * v
                    row = {"line": line, "year": vy, "config": str(cfg), "glm_weight": wgt, "trees": n, "k": k,
                           "hold_dev": hold_dev(hold.target.to_numpy(), hr, hold.weight.to_numpy()),
                           **metrics(P[vy]["va"], vr * P[vy]["va"].weight.to_numpy())}
                    save(row, "9_grid.csv")
                    res[(str(cfg), wgt, vy)] = (row, v, h)
                log(f"11k {line} {vy} {cfg}: {n} trees, k {k:.3f}, {time.time() - t0:.0f}s")
        # 2. choose config and weight on the hold year (mean over folds)
        g = pd.DataFrame([r for (r, _, _) in res.values()])
        pick = g.groupby(["config", "glm_weight"]).hold_dev.mean().idxmin()
        cfg_s, wgt = pick
        log(f"11k {line}: chosen on hold year -> {cfg_s}, GLM weight {wgt}")
        # 3. seed bagging for the chosen config
        params = dict(PARAMS2, **eval(cfg_s))
        rows = []
        for vy in P:
            vs = [res[(cfg_s, wgt, vy)][1]]
            for s in SEEDS[1:]:
                _, v, _, _ = gbm(P[vy], feats, params, s)
                vs.append(v)
            v_bag = np.mean(vs, axis=0)
            w_va = P[vy]["va"].weight.to_numpy()
            for name, vr in [("chosen, 1 seed", wgt * P[vy]["g_va"] + (1 - wgt) * vs[0]),
                             ("chosen, 3 seeds", wgt * P[vy]["g_va"] + (1 - wgt) * v_bag)]:
                rows.append({"line": line, "year": vy, "model": name, **metrics(P[vy]["va"], vr * w_va)})
            rows.append({"line": line, "year": vy, "model": "GLM (v6 features)", **glm_m[vy]})
        r = pd.DataFrame(rows)
        for row in rows:
            save(dict(row, config=cfg_s, glm_weight=wgt), "9_chosen.csv")
        base = r[r.model == "GLM (v6 features)"].set_index("year")
        for name in ["chosen, 1 seed", "chosen, 3 seeds"]:
            c = r[r.model == name].set_index("year")
            gain = c.D2.mean() - base.D2.mean()
            tails = all(tails_ok(c.loc[y], base.loc[y]) for y in c.index)
            summary.append({"line": line, "model": name, "config": cfg_s, "glm_weight": wgt, "gain": gain,
                            "tails": "ok" if tails else "fail", "low": c.low.round(3).tolist(),
                            "top": c.top.round(3).tolist(), "glm_low": base.low.round(3).tolist(),
                            "glm_top": base.top.round(3).tolist(),
                            "verdict": "ADOPT" if gain >= 0.003 and tails else "no"})
        pd.DataFrame(summary).to_csv(OUT / "9_summary.csv", index=False)
    except Exception:
        log(f"11k FAILED {line}\n{traceback.format_exc()}")

pd.set_option("display.width", 250); pd.set_option("display.max_colwidth", 60)
log("11k summary (config and GLM weight chosen on the hold year; rule +0.003 and tails within 0.1):\n"
    + pd.DataFrame(summary).round(4).to_string(index=False))
log("=== 11k all done ===")

20:35:49 === 11k start ===
20:37:01 11k motor 2023 {'num_leaves': 7, 'min_data_in_leaf': 1000, 'lambda_l2': 1.0}: 513 trees, k 1.048, 9s
20:37:14 11k motor 2024 {'num_leaves': 7, 'min_data_in_leaf': 1000, 'lambda_l2': 1.0}: 455 trees, k 1.038, 13s
20:37:24 11k motor 2023 {'num_leaves': 7, 'min_data_in_leaf': 1000, 'lambda_l2': 10.0}: 513 trees, k 1.048, 10s
20:37:38 11k motor 2024 {'num_leaves': 7, 'min_data_in_leaf': 1000, 'lambda_l2': 10.0}: 455 trees, k 1.038, 14s
20:37:48 11k motor 2023 {'num_leaves': 7, 'min_data_in_leaf': 1000, 'lambda_l2': 50.0}: 508 trees, k 1.047, 10s
20:38:01 11k motor 2024 {'num_leaves': 7, 'min_data_in_leaf': 1000, 'lambda_l2': 50.0}: 445 trees, k 1.038, 13s
20:38:11 11k motor 2023 {'num_leaves': 7, 'min_data_in_leaf': 5000, 'lambda_l2': 1.0}: 431 trees, k 1.041, 9s
20:38:24 11k motor 2024 {'num_leaves': 7, 'min_data_in_leaf': 5000, 'lambda_l2': 1.0}: 444 trees, k 1.035, 14s
20:38:34 11k motor 2023 {'num_leaves': 7, 'min_data_in_leaf': 5000, 'lambda_l2': 10

In [9]:
# 11l v7 = v6 with contents replaced by LightGBM alone (num_leaves 7, min_data_in_leaf 5000, lambda_l2 50),
# averaged over 3 seeds; config and GLM weight (0.0) chosen on the hold year in 11k, passed the pre-set rule.
line = "contents"
df, loading = data[line]["df"], data[line]["loading"]
feats = [c for c in feats_of(df, line) if c not in EXTRA_DROP[line]]
cfg = dict(num_leaves=7, min_data_in_leaf=5000, lambda_l2=50.0)

p, pt = load_policies(line, "train"), load_policies(line, "test")
Ft = add_history(build_features(line, "test", policies=pt), pt, [p, pt], claims)
assert list(Ft.row_id) == list(pt.row_id)
rates, info = [], []
for s in SEEDS:
    model, cats, k, n_trees = fit_gbm2(df, feats, params=dict(PARAMS2, **cfg, seed=s))
    Xt, _ = prep2(Ft[feats], cats)
    rates.append(k * model.predict(Xt))
    info.append(f"seed {s}: {n_trees} trees, k {k:.3f}")
rate = np.mean(rates, axis=0)
pred = np.maximum((rate + loading) * pt.exposure_clean.to_numpy(), 0)

v6 = pd.read_csv("../submissions/v6.csv")
sub = v6.set_index("row_id")
old = sub.loc[Ft.row_id, "prediction"].to_numpy()
sub.loc[Ft.row_id, "prediction"] = pred
sub = sub.reset_index()
assert len(sub) == len(v6) and (sub.row_id == v6.row_id).all()
assert sub.prediction.notna().all() and (sub.prediction >= 0).all()
print("; ".join(info))
print(f"contents total v7 / v6 = {pred.sum() / old.sum():.3f}; "
      f"rows changed by > 2x: {((pred / old > 2) | (pred / old < 0.5)).mean():.2%}; "
      f"seed spread (mean abs diff between seeds / mean): {np.mean(np.abs(rates[0] - rates[1])) / rate.mean():.2%}")
sub.to_csv("../submissions/v7.csv", index=False)
print("wrote submissions/v7.csv")

seed 42: 418 trees, k 1.056; seed 7: 396 trees, k 1.057; seed 2026: 419 trees, k 1.057
contents total v7 / v6 = 1.007; rows changed by > 2x: 0.02%; seed spread (mean abs diff between seeds / mean): 5.76%
wrote submissions/v7.csv


In [10]:
# 11m Tweedie power used for FITTING the GLM (scoring stays at p = 1.5)
# Rule fixed before results: per line, adopt the p with the highest average D2 if it is >= 0.0005 above p = 1.5
# and both tails are no more than 0.1 further from 1 than at p = 1.5; otherwise keep 1.5.
POWERS = [1.3, 1.4, 1.5, 1.6, 1.7]
rows = []
for line in LINES:
    df = data[line]["df"]
    feats = [c for c in feats_of(df, line) if c not in EXTRA_DROP[line]]
    for vy, tys in FOLDS:
        t0 = time.time()
        tr, va = df[df.year.isin(tys)], df[df.year == vy]
        for pw in POWERS:
            glm = BandedTweedieGLM(**dict(GLM_CONFIG[line], power=pw)).fit(tr[feats], tr.target / tr.weight, tr.weight)
            pred = glm.predict_rate(va[feats]) * va.weight.to_numpy()
            rows.append({"line": line, "year": vy, "power": pw, "k": glm.k_, **metrics(va, pred)})
        log(f"11m {line} {vy}: {time.time() - t0:.0f}s")

pw_res = pd.DataFrame(rows)
pw_res.to_csv(OUT / "10_tweedie_power.csv", index=False)
for line in LINES:
    t = pw_res[pw_res.line == line].pivot(index="power", columns="year", values=["D2", "A/E", "low", "top", "k"])
    t.columns = [f"{m}_{y}" for m, y in t.columns]
    t["D2_avg"] = t[["D2_2023", "D2_2024"]].mean(axis=1)
    print(f"\n===== {line} (D2 always scored at p = 1.5) =====")
    print(t.round(4).to_string())

print("\nrule: adopt p if D2 >= p1.5 + 0.0005 and tails within 0.1 of p1.5")
for line in LINES:
    s = pw_res[pw_res.line == line]
    base = s[s.power == 1.5].set_index("year")
    best, best_gain = 1.5, 0.0
    for pw in POWERS:
        c = s[s.power == pw].set_index("year")
        gain = c.D2.mean() - base.D2.mean()
        tails = all(abs(c.loc[y, m] - 1) <= abs(base.loc[y, m] - 1) + 0.1 for y in c.index for m in ["low", "top"])
        if pw != 1.5 and gain >= 0.0005 and tails and gain > best_gain:
            best, best_gain = pw, gain
        print(f"  {line:9s} p={pw}: gain {gain:+.4f}, tails {'ok' if tails else 'fail'}")
    print(f"  -> {line}: use p = {best}" + (f" (+{best_gain:.4f})" if best != 1.5 else " (no change)"))

21:05:55 11m motor 2023: 77s
21:07:51 11m motor 2024: 116s
21:08:45 11m buildings 2023: 54s
21:10:03 11m buildings 2024: 78s
21:11:00 11m contents 2023: 56s
21:12:22 11m contents 2024: 83s

===== motor (D2 always scored at p = 1.5) =====
       D2_2023  D2_2024  A/E_2023  A/E_2024  low_2023  low_2024  top_2023  top_2024  k_2023  k_2024  D2_avg
power                                                                                                      
1.3     0.0520   0.0556    1.0312    0.9953    1.0626    1.0443    0.9761    0.9653  1.0048  1.0044  0.0538
1.4     0.0523   0.0554    1.0307    0.9987    0.9893    1.0143    0.9974    1.0036  1.0105  1.0096  0.0538
1.5     0.0522   0.0546    1.0296    1.0027    0.9071    0.9837    1.0392    1.0557  1.0202  1.0189  0.0534
1.6     0.0513   0.0529    1.0277    1.0066    0.8595    0.9075    1.1057    1.1288  1.0348  1.0330  0.0521
1.7     0.0490   0.0497    1.0256    1.0100    0.7986    0.8288    1.1761    1.2202  1.0527  1.0507  0.0493

=====

In [11]:
# 11n Bodily injury severity trend after the 2023 reform (currently 0%)
# (1) model-free: BI severity change 2021->2022 (pre-reform) and 2023->2024 (post-reform, equal maturity:
#     losses in Jan-Jun, reported within 180 days, gross of excess, capped at 500k);
# (2) rebuild the motor target with BI trend 0 / 2 / 4 / 6% and refit the v6 motor GLM on both folds:
#     with the right trend, earlier years brought to 2025 level should match the next year (A/E near 1).
# Rule fixed before results: take the candidate closest to the post-reform model-free estimate; adopt it if
# the motor validation A/E moves toward 1 in BOTH years compared with 0%; otherwise keep 0%.
import copy
from src.target import ADJ, build_model_target, match_claims

p = load_policies("motor", "train")
m = match_claims(claims, p, "motor", cap=None)
bi = m[m.peril == "bodily_injury"].copy()
bi["gross_cap"] = np.minimum(bi.incurred_to_date, ADJ["motor"]["cap"])
bi["lag"] = (bi.report_date - bi.loss_date).dt.days
eq = bi[(bi.loss_date.dt.month <= 6) & (bi.lag <= 180)]
s = eq.groupby("loss_year").gross_cap.agg(["count", "mean", "median"])
print("BI severity, losses Jan-Jun reported within 180 days (gross of excess, capped):")
print(s.round(0).to_string())
pre = s.loc[2022, "mean"] / s.loc[2021, "mean"] - 1
post = s.loc[2024, "mean"] / s.loc[2023, "mean"] - 1
post_med = s.loc[2024, "median"] / s.loc[2023, "median"] - 1
print(f"\npre-reform 2021->2022: mean {pre:+.1%}; post-reform 2023->2024: mean {post:+.1%}, median {post_med:+.1%}")

CANDS = [0.0, 0.02, 0.04, 0.06]
target_t = min(CANDS, key=lambda t: abs(t - post))
print(f"candidate closest to the post-reform estimate: {target_t:.0%}")

feats_df = data["motor"]["df"].drop(columns=["year", "exposure_clean", "target", "weight"])
feats = [c for c in feats_of(data["motor"]["df"], "motor") if c not in EXTRA_DROP["motor"]]
rows = []
for t in CANDS:
    adj = copy.deepcopy(ADJ["motor"]); adj["sev_trend"]["bodily_injury"] = t
    mt, comp, loading = build_model_target(claims, p, "motor", adj)
    df = mt.merge(feats_df, on="row_id", how="left", validate="one_to_one")
    lvl = df.groupby("year").apply(lambda g: g.target.sum() / g.weight.sum())
    for vy, tys in FOLDS:
        t0 = time.time()
        tr, va = df[df.year.isin(tys)], df[df.year == vy]
        glm = BandedTweedieGLM(**GLM_CONFIG["motor"]).fit(tr[feats], tr.target / tr.weight, tr.weight)
        pred = glm.predict_rate(va[feats]) * va.weight.to_numpy()
        rows.append({"bi_trend": t, "year": vy, **metrics(va, pred),
                     "level_by_year": " / ".join(f"{v:.0f}" for v in lvl.values)})
        log(f"11n BI trend {t:.0%} {vy}: {time.time() - t0:.0f}s")

bt = pd.DataFrame(rows)
bt.to_csv(OUT / "11_bi_trend.csv", index=False)
print("\nmotor GLM (v6 features) by BI trend assumption:")
print(bt.pivot(index="bi_trend", columns="year", values=["A/E", "D2", "low", "top"]).round(4).to_string())
print("\ncost per weight by year 2021-24 (should be level if adjustments are right):")
print(bt.drop_duplicates("bi_trend")[["bi_trend", "level_by_year"]].to_string(index=False))

ae = bt.pivot(index="bi_trend", columns="year", values="A/E")
toward = all(abs(ae.loc[target_t, y] - 1) < abs(ae.loc[0.0, y] - 1) for y in ae.columns)
print(f"\nrule: candidate {target_t:.0%}; A/E closer to 1 than at 0% in both years -> "
      f"{'adopt ' + format(target_t, '.0%') if (target_t != 0.0 and toward) else 'keep 0%'}")

BI severity, losses Jan-Jun reported within 180 days (gross of excess, capped):
           count     mean   median
loss_year                         
2021         625  20414.0  13269.0
2022         641  21620.0  12380.0
2023         660  18125.0   9617.0
2024         611  19528.0  10525.0

pre-reform 2021->2022: mean +5.9%; post-reform 2023->2024: mean +7.7%, median +9.4%
candidate closest to the post-reform estimate: 6%
21:17:45 11n BI trend 0% 2023: 16s
21:18:09 11n BI trend 0% 2024: 24s
21:18:24 11n BI trend 2% 2023: 15s
21:18:48 11n BI trend 2% 2024: 24s
21:19:04 11n BI trend 4% 2023: 15s
21:19:27 11n BI trend 4% 2024: 23s
21:19:43 11n BI trend 6% 2023: 15s
21:20:06 11n BI trend 6% 2024: 23s

motor GLM (v6 features) by BI trend assumption:
             A/E              D2             low             top        
year        2023    2024    2023    2024    2023    2024    2023    2024
bi_trend                                                                
0.00      1.0296  1.0027  0